In [0]:
from databricks.sdk import WorkspaceClient
import json

w = WorkspaceClient()

# ============================================================
# DYNAMIC PATH RESOLUTION
# ============================================================
# The base_path and experiment_name are resolved from the current
# user's identity so this notebook works for any user without
# manual edits. The notebooks (01_generate_data, 02_train_model,
# 03_batch_inference) must live under:
#   /Users/<your_email>/mlops-jobs/
# If your notebooks are in a different location, override base_path below.
current_user = spark.sql("SELECT current_user()").first()[0]
base_path = f"/Users/{current_user}/mlops-jobs"

# ============================================================
# PIPELINE PARAMETERS — single source of truth
# ============================================================
# Adjust these values as needed for your environment.
#
# CATALOG NOTE:
#   If your workspace has "Default Storage" enabled (no metastore
#   storage root URL configured), you CANNOT create new catalogs
#   via SQL. You must either:
#     1. Use an existing catalog (like "agents" below).
#     2. Create the catalog via the Databricks UI (which uses
#        Default Storage automatically).
#     3. Provide a MANAGED LOCATION in your CREATE CATALOG statement:
#          CREATE CATALOG my_catalog MANAGED LOCATION 's3://bucket/path'
#   If you have a metastore storage root configured, any catalog
#   name will work — the 01_generate_data notebook handles creation.
catalog = "agents"
schema = "mlops_demo"

# Data generation
table_name = "synthetic_training_data"
n_samples = "10000"       # Number of rows to generate
n_features = "10"         # Number of feature columns

# Model training
model_name = "synthetic_classifier"
n_estimators = "100"      # RandomForest: number of trees
max_depth = "10"          # RandomForest: max tree depth
experiment_name = f"/Users/{current_user}/mlops-jobs/synthetic_classifier_experiment"

# Batch inference
output_table = "prediction_results"
model_alias = "Champion"  # UC model alias to load for scoring

# ============================================================
# INTER-TASK COMMUNICATION (via dbutils.jobs.taskValues)
# ============================================================
# Tasks pass values downstream at runtime so you don't need to
# duplicate parameters that are produced by earlier tasks:
#
#   generate_data SETS:
#     full_table_name, row_count, n_features
#         |
#         v
#   train_model GETS full_table_name from generate_data
#   train_model SETS:
#     full_model_name, model_version, accuracy, run_id
#         |
#         v
#   batch_inference GETS full_model_name from train_model
#   batch_inference GETS full_table_name from generate_data
# ============================================================

job_config = {
    "name": "mlops_demo_pipeline",
    "tasks": [
        {
            "task_key": "generate_data",
            "notebook_task": {
                "notebook_path": f"{base_path}/01_generate_data",
                "source": "WORKSPACE",
                "base_parameters": {
                    "catalog": catalog,
                    "schema": schema,
                    "table_name": table_name,
                    "n_samples": n_samples,
                    "n_features": n_features,
                },
            },
            "environment_key": "Default",
        },
        {
            "task_key": "train_model",
            "notebook_task": {
                "notebook_path": f"{base_path}/02_train_model",
                "source": "WORKSPACE",
                "base_parameters": {
                    "catalog": catalog,
                    "schema": schema,
                    "model_name": model_name,
                    "n_estimators": n_estimators,
                    "max_depth": max_depth,
                    "experiment_name": experiment_name,
                },
            },
            "environment_key": "Default",
            "depends_on": [{"task_key": "generate_data"}],
        },
        {
            "task_key": "batch_inference",
            "notebook_task": {
                "notebook_path": f"{base_path}/03_batch_inference",
                "source": "WORKSPACE",
                "base_parameters": {
                    "catalog": catalog,
                    "schema": schema,
                    "model_alias": model_alias,
                    "output_table": output_table,
                },
            },
            "environment_key": "Default",
            "depends_on": [{"task_key": "train_model"}],
        },
    ],
    "environments": [
        {
            "environment_key": "Default",
            "spec": {
                "client": "1",
                "dependencies": [
                    "mlflow[databricks]",
                    "scikit-learn",
                ],
            },
        }
    ],
}

# ============================================================
# JOB CREATION
# ============================================================
# If a job_id variable exists from a previous run of this cell,
# the old job is deleted first to avoid duplicates. On first run
# the delete is silently skipped.
try:
    w.api_client.do("POST", "/api/2.1/jobs/delete", body={"job_id": job_id})
    print(f"Deleted previous job: {job_id}")
except:
    pass

# Create job via API
response = w.api_client.do("POST", "/api/2.1/jobs/create", body=job_config)
job_id = response["job_id"]

print(f"Created job: {job_id}")
print(f"Job URL: {w.config.host}/jobs/{job_id}")

In [0]:
# Trigger the job
response = w.api_client.do("POST", "/api/2.1/jobs/run-now", body={"job_id": job_id})
run_id = response["run_id"]
print(f"Triggered run: {run_id}")
print(f"Run URL: {w.config.host}/jobs/{job_id}/runs/{run_id}")

In [0]:
import time

def poll_run(client, run_id, poll_interval=30):
    """Poll job run until terminal state, printing per-task progress."""
    terminal_states = {"TERMINATED", "SKIPPED", "INTERNAL_ERROR"}
    task_states = {}

    while True:
        run_status = client.do("GET", f"/api/2.1/jobs/runs/get", query={"run_id": run_id})
        life_cycle = run_status["state"]["life_cycle_state"]

        # Print per-task updates
        for task in run_status.get("tasks", []):
            task_key = task["task_key"]
            task_state = task.get("state", {}).get("life_cycle_state", "PENDING")
            result_state = task.get("state", {}).get("result_state", "-")

            current = f"{task_state} ({result_state})"
            if task_states.get(task_key) != current:
                task_states[task_key] = current
                print(f"  [{task_key}] {task_state} | result: {result_state}")

        # Check if the overall run is done
        if life_cycle in terminal_states:
            result = run_status["state"].get("result_state", "UNKNOWN")
            print(f"\nRun finished: {life_cycle} | Result: {result}")

            # Print duration per task
            print("\nTask durations:")
            for task in run_status.get("tasks", []):
                start = task.get("start_time")
                end = task.get("end_time")
                if start and end:
                    duration_s = (end - start) / 1000
                    print(f"  {task['task_key']}: {duration_s:.1f}s")
            return run_status

        time.sleep(poll_interval)

print(f"Polling run {run_id} every 30s...\n")
final_status = poll_run(w.api_client, run_id)

In [0]:
import json

# Retrieve and parse structured outputs from each task
task_outputs = {}

for task in final_status.get("tasks", []):
    task_key = task["task_key"]
    task_run_id = task["run_id"]

    print(f"\n{'='*60}")
    print(f"Task: {task_key}")
    print(f"{'='*60}")

    try:
        output = w.api_client.do("GET", "/api/2.1/jobs/runs/get-output", query={"run_id": task_run_id})

        notebook_output = output.get("notebook_output", {})
        if notebook_output.get("result"):
            result = json.loads(notebook_output["result"])
            task_outputs[task_key] = result
            for key, value in result.items():
                print(f"  {key}: {value}")

        if output.get("error"):
            print(f"  ERROR: {output['error']}")
        if output.get("error_trace"):
            print(f"  Traceback:\n{output['error_trace']}")

    except Exception as e:
        print(f"  Could not retrieve output: {e}")

# Summary
print(f"\n{'='*60}")
print("Pipeline Summary")
print(f"{'='*60}")
if "generate_data" in task_outputs:
    print(f"  Data: {task_outputs['generate_data'].get('rows')} rows -> {task_outputs['generate_data'].get('table')}")
if "train_model" in task_outputs:
    print(f"  Model: {task_outputs['train_model'].get('model')} v{task_outputs['train_model'].get('version')} (accuracy={task_outputs['train_model'].get('accuracy')})")
if "batch_inference" in task_outputs:
    print(f"  Scored: {task_outputs['batch_inference'].get('rows_scored')} rows -> {task_outputs['batch_inference'].get('output_table')}")